# Transformação e limpeza de dados com pandas
Transformação de dados semiestruturados (JSON aninhado, listas em colunas, números e datas em texto, extração com expressões regulares) e limpeza de uma base de clientes de telecomunicações: valores em branco, nulos, duplicados, *outliers* e codificação de variáveis categóricas.

**Origem:** cursos Alura (transformação e manipulação de dados; limpeza e tratamento de dados).

**Dados:** enviar para `/content/` no Colab: `dados_hospedagem.json`, `moveis_disponiveis.json`, `dados_locacao_imoveis.json`, `banco_clientes.csv` (seções 1–4) e `dataset-telecon.json` (seções 5–9).

**Índice**
1. Dados de hospedagem: JSON aninhado, números e textos
2. Datas: disponibilidade de imóveis
3. Locação de imóveis: várias colunas com listas
4. Cadastro de clientes: expressões regulares
5. Base de telecomunicações: carga e valores em branco
6. Linhas vazias, duplicadas e valores nulos
7. Exemplo: tratamento de um JSON de cursos
8. *Outliers*
9. Variáveis categóricas

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 1. Dados de hospedagem: JSON aninhado, números e textos
O arquivo tem uma chave `info_moveis` com os registros. Depois do `json_normalize`, várias colunas contêm **listas** (um valor por imóvel do anúncio); `explode` transforma cada elemento da lista em uma linha.

In [ ]:
hospedagens = pd.read_json('/content/dados_hospedagem.json')
hospedagens = pd.json_normalize(hospedagens['info_moveis'])

colunas = list(hospedagens.columns)
colunas

In [ ]:
hospedagens = hospedagens.explode(colunas[3:])
hospedagens.reset_index(inplace=True, drop=True)
hospedagens.info()

### 1.1 Manipulação de números
Após o `explode` as colunas ficam com tipo `object`; elas são convertidas com `astype`. Valores monetários vêm como texto (`$1,250.00`): remove-se o `$` e a vírgula de milhar antes de converter para `float`.

In [ ]:
hospedagens['max_hospedes'] = hospedagens['max_hospedes'].astype(np.int64)
hospedagens['avaliacao_geral'] = hospedagens['avaliacao_geral'].astype(np.float64)

col_numericas = ['quantidade_banheiros', 'quantidade_quartos', 'quantidade_camas']
hospedagens[col_numericas] = hospedagens[col_numericas].astype(np.int64)

In [ ]:
col_monetarias = ['preco', 'taxa_deposito', 'taxa_limpeza']
hospedagens[col_monetarias] = (
    hospedagens[col_monetarias]
    .map(lambda x: x.replace('$', '').replace(',', '').strip())
    .astype(np.float64)
)
hospedagens[col_monetarias].describe()

### 1.2 Manipulação de textos
Para preparar descrições para análise de texto: minúsculas, remoção de caracteres especiais com *regex* (mantendo letras, números, hífen e apóstrofo), remoção de hífens soltos (`(?<!\w)-(?!\w)`: hífen sem letra antes e depois) e separação em lista de palavras com `str.split()`.

In [ ]:
hospedagens['descricao_local'] = (
    hospedagens['descricao_local']
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9\-']", ' ', regex=True)
    .str.replace(r'(?<!\w)-(?!\w)', ' ', regex=True)
    .str.split()
)

hospedagens['descricao_vizinhanca'] = (
    hospedagens['descricao_vizinhanca']
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9\-']", ' ', regex=True)
    .str.replace(r'(?<!\w)-(?!\w)', ' ', regex=True)
    .str.split()
)

In [ ]:
# Comodidades vêm como texto "{Wifi,"TV a cabo",...}": remove chaves e aspas e separa pela vírgula
hospedagens['comodidades'] = hospedagens['comodidades'].str.replace(r'\{|}|"', '', regex=True).str.split(',')
hospedagens.head()

## 2. Datas: disponibilidade de imóveis
`pd.to_datetime` converte o texto em data; com o acessor `.dt` é possível extrair partes da data ou formatá-la (`strftime('%Y-%m')`) para agrupar por mês.

In [ ]:
dt_data = pd.read_json('/content/moveis_disponiveis.json')
dt_data['data'] = pd.to_datetime(dt_data['data'])

dt_data['preco'] = dt_data['preco'].fillna('0.0')
dt_data['preco'] = dt_data['preco'].apply(lambda x: x.replace('$', '').replace(',', '')).astype(np.float64)

In [ ]:
# Total de vagas disponíveis por mês
subset = dt_data.groupby(dt_data['data'].dt.strftime('%Y-%m'))['vaga_disponivel'].sum()
subset

## 3. Locação de imóveis: várias colunas com listas
`explode` aceita uma lista de colunas, desde que as listas de cada linha tenham o mesmo tamanho.

In [ ]:
loc_imoveis = pd.read_json('/content/dados_locacao_imoveis.json')
loc_imoveis = pd.json_normalize(loc_imoveis['dados_locacao'])
loc_imoveis = loc_imoveis.explode(['datas_combinadas_pagamento', 'datas_de_pagamento', 'valor_aluguel'])

loc_imoveis['valor_aluguel'] = loc_imoveis['valor_aluguel'].apply(
    lambda x: x.replace('$', '').replace(',', '').replace('reais', '').strip()
).astype(np.float64)

loc_imoveis['apartamento'] = loc_imoveis['apartamento'].str.replace('(blocoAP)', '', regex=False)
loc_imoveis

## 4. Cadastro de clientes: expressões regulares
- `str.extract(r'(\d{5}-\d{3})')` captura o CEP do endereço;
- `str.split(r'\s(?=\d+$)', expand=True)` separa rua e número no último espaço seguido apenas de dígitos.

In [ ]:
clientes = pd.read_csv('/content/banco_clientes.csv', sep=',')

clientes['nome'] = clientes['nome'].str.upper()
clientes['telefone'] = clientes['telefone'].replace(r'\(|\)', '', regex=True)

clientes['cep'] = clientes['endereco'].str.extract(r'(\d{5}-\d{3})')
clientes['endereco'] = clientes['endereco'].str.replace(r'(\d{5}-\d{3})', '', regex=True).str.strip()
clientes[['rua', 'numero']] = clientes['endereco'].str.replace(',', '').str.split(r'\s(?=\d+$)', regex=True, expand=True)

clientes = clientes.drop(['endereco'], axis=1)
clientes.head()

## 5. Base de telecomunicações: carga e valores em branco
Lido com `pd.read_json`, o arquivo fica com colunas que contêm dicionários (`cliente`, `conta`, `telefone`...). Lendo com o módulo `json` e aplicando `json_normalize`, cada campo interno vira uma coluna (`conta.cobranca.Total`).

In [ ]:
df = pd.read_json('/content/dataset-telecon.json')
pd.json_normalize(df['conta']).head()

In [ ]:
with open('/content/dataset-telecon.json') as arquivo:
    json_data = json.load(arquivo)

df_norm = pd.json_normalize(json_data)
df_norm.info()

In [ ]:
def mostrar_valores_unicos(dados):
    for col in dados.columns:
        print(f'Coluna: {col}')
        print(dados[col].unique())
        print('-' * 30)


mostrar_valores_unicos(df_norm)

### 5.1 Totais em branco
A coluna `conta.cobranca.Total` tem textos `' '` (por isso ela é `object`). Nesses casos o contrato é de 24 meses: o tempo de serviço é preenchido com 24 e o total com cobrança mensal × tempo de serviço; depois a coluna é convertida para `float`.

In [ ]:
colunas_cobranca = ['conta.cobranca.mensal', 'conta.contrato', 'cliente.tempo_servico', 'conta.cobranca.Total']
df_norm[df_norm['conta.cobranca.Total'] == ' '][colunas_cobranca]

In [ ]:
idx = df_norm[df_norm['conta.cobranca.Total'] == ' '].index

df_norm.loc[idx, 'cliente.tempo_servico'] = 24
df_norm.loc[idx, 'conta.cobranca.Total'] = df_norm.loc[idx, 'conta.cobranca.mensal'] * df_norm.loc[idx, 'cliente.tempo_servico']

df_norm['conta.cobranca.Total'] = df_norm['conta.cobranca.Total'].astype(float)
df_norm.loc[idx, colunas_cobranca]

## 6. Linhas vazias, duplicadas e valores nulos
Linhas sem a variável-alvo `Churn` (texto vazio) são descartadas; depois removem-se as linhas duplicadas e tratam-se os nulos.

In [ ]:
df_norm.query("Churn == ''")

In [ ]:
df_novo = df_norm[df_norm['Churn'] != ''].copy()
df_novo.reset_index(drop=True, inplace=True)

print('Duplicadas:', df_novo.duplicated().sum())
df_novo.drop_duplicates(inplace=True)
df_novo.reset_index(drop=True, inplace=True)

In [ ]:
print(df_novo.isna().sum())
print('Total de nulos:', df_novo.isna().sum().sum())
df_novo[df_novo.isna().any(axis=1)]

### 6.1 Preenchendo e removendo nulos
O tempo de serviço ausente é estimado como `ceil(total / mensal)`. Já as linhas sem contrato, faturamento eletrônico ou método de pagamento são removidas com `dropna(subset=...)`.

In [ ]:
filtro = df_novo['cliente.tempo_servico'].isna()
df_novo[filtro][['cliente.tempo_servico', 'conta.cobranca.mensal', 'conta.cobranca.Total']]

In [ ]:
df_novo['cliente.tempo_servico'] = df_novo['cliente.tempo_servico'].fillna(
    np.ceil(df_novo['conta.cobranca.Total'] / df_novo['conta.cobranca.mensal'])
)

dropar = ['conta.contrato', 'conta.faturamente_eletronico', 'conta.metodo_pagamento']
print('Linhas com nulos nessas colunas:', df_novo[dropar].isna().any(axis=1).sum())

df = df_novo.dropna(subset=dropar).copy()
df.reset_index(drop=True, inplace=True)
df.isna().sum()

## 7. Exemplo: tratamento de um JSON de cursos
Exercício com dados fictícios que reúne os passos anteriores: remover duplicados, transformar textos vazios em nulos (`pd.NA`), remover linhas incompletas e converter os tipos.

In [ ]:
cursos_json = [
    {"curso": "Introdução à programação", "categoria": "Programação", "carga_horaria": "20 horas",
     "concluintes": 100, "data_inicio": "2022-01-01", "data_conclusao": "2022-01-20",
     "descricao": "Curso introdutório à programação com Python", "preco": "99.90",
     "instrutor": {"nome": "João Silva", "email": "joao.silva@emailaleatorio.com", "telefone": "(11) 9999-9999"}},
    {"curso": "Excel para iniciantes", "categoria": "Produtividade", "carga_horaria": None,
     "concluintes": None, "data_inicio": None, "data_conclusao": None,
     "descricao": None, "preco": None,
     "instrutor": {"nome": "Maria Oliveira", "email": "maria.oliveira@emailaleatorio.com", "telefone": "(11) 8888-8888"}},
    {"curso": "Marketing digital para negócios", "categoria": "Marketing", "carga_horaria": "30 horas",
     "concluintes": 75, "data_inicio": "2022-03-01", "data_conclusao": "2022-03-31",
     "descricao": "Curso introdutório em marketing", "preco": 89.9,
     "instrutor": {"nome": "Ana Santos", "email": "ana.santos@emailaleatorio.com", "telefone": "(11) 7777-7777"}},
    {"curso": "Inteligência artificial", "categoria": "Programação", "carga_horaria": "40 horas",
     "concluintes": "", "data_inicio": "2022-04-01", "data_conclusao": "",
     "descricao": "Curso avançado sobre inteligência artificial com Python", "preco": 129.9,
     "instrutor": {"nome": "", "email": "contato@emailaleatorio.com", "telefone": ""}},
    {"curso": "Inglês para negócios", "categoria": "Idiomas", "carga_horaria": "20 horas",
     "concluintes": 30, "data_inicio": "", "data_conclusao": "",
     "descricao": "Curso de inglês para negócios", "preco": 69.9,
     "instrutor": {"nome": "John Smith", "email": "", "telefone": ""}},
    {"curso": "Introdução à programação", "categoria": "Programação", "carga_horaria": "20 horas",
     "concluintes": 100, "data_inicio": "2022-01-01", "data_conclusao": "2022-01-20",
     "descricao": "Curso introdutório à programação com Python", "preco": "99.90",
     "instrutor": {"nome": "João Silva", "email": "joao.silva@emailaleatorio.com", "telefone": "(11) 9999-9999"}}
]

cursos = pd.json_normalize(cursos_json)
print('Duplicadas:', cursos.duplicated().sum())
cursos = cursos.drop_duplicates()
cursos.isna().sum()

In [ ]:
cursos = cursos.replace('', pd.NA).dropna()

cursos['concluintes'] = cursos['concluintes'].astype(int)
cursos['data_inicio'] = pd.to_datetime(cursos['data_inicio'])
cursos['data_conclusao'] = pd.to_datetime(cursos['data_conclusao'])
cursos['preco'] = cursos['preco'].astype(float)
cursos.info()

## 8. *Outliers*
### 8.1 Método do intervalo interquartil (IQR)
$IQR = Q_3 - Q_1$; são *outliers* os valores abaixo de $Q_1 - 1{,}5 \cdot IQR$ ou acima de $Q_3 + 1{,}5 \cdot IQR$. No tempo de serviço, os *outliers* são substituídos pela estimativa `ceil(total / mensal)`.

In [ ]:
def limites_iqr(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR


sns.boxplot(x=df['cliente.tempo_servico'])
plt.show()

limite_inferior, limite_superior = limites_iqr(df['cliente.tempo_servico'])
outliers = (df['cliente.tempo_servico'] < limite_inferior) | (df['cliente.tempo_servico'] > limite_superior)
print(f'Limites: {limite_inferior:.1f} a {limite_superior:.1f}; outliers: {outliers.sum()}')

In [ ]:
df.loc[outliers, 'cliente.tempo_servico'] = np.ceil(
    df.loc[outliers, 'conta.cobranca.Total'] / df.loc[outliers, 'conta.cobranca.mensal']
)
sns.boxplot(x=df['cliente.tempo_servico'])
plt.show()

### 8.2 Removendo os *outliers* restantes
Os limites são recalculados após a substituição e as linhas que continuam fora deles são removidas com a máscara negada (`~`).

In [ ]:
limite_inferior, limite_superior = limites_iqr(df['cliente.tempo_servico'])
outliers = (df['cliente.tempo_servico'] < limite_inferior) | (df['cliente.tempo_servico'] > limite_superior)

df = df[~outliers].reset_index(drop=True)
df.shape

### 8.3 Função reutilizável e exemplo com transações
A função `calc_outlier` devolve as linhas que são *outliers* pelo critério IQR em qualquer coluna de um DataFrame.

In [ ]:
def calc_outlier(column, dataframe):
    limite_inferior, limite_superior = limites_iqr(dataframe[column])
    return dataframe[(dataframe[column] < limite_inferior) | (dataframe[column] > limite_superior)]


# DataFrame com 30 transações de exemplo
treino1 = pd.DataFrame({
    'ID da transação': range(1, 31),
    'Valor da transação': [100, 200, 150, 500, 300, 913, 250, 400, 200, 150,
                           200, 200, 400, 300, 150, 301, 805, 300, 400, 250,
                           150, 100, 500, 600, 200, 350, 100, 250, 800, 250],
    'Data da transação': pd.date_range(start='2022-01-01', end='2022-01-30', freq='D'),
    'Local da transação': ['São Paulo, Brasil', 'Rio de Janeiro, Brasil', 'Belo Horizonte, Brasil', 'São Paulo, Brasil',
                           'São Paulo, Brasil', 'Nova Iorque, EUA', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil',
                           'Rio de Janeiro, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil',
                           'São Paulo, Brasil', 'São Paulo, Brasil', 'Los Angeles, EUA', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil',
                           'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil', 'São Paulo, Brasil',
                           'São Paulo, Brasil', 'São Paulo, Brasil', 'Miami, EUA', 'São Paulo, Brasil']
})

calc_outlier('Valor da transação', treino1)

### 8.4 Método do *z-score*
$z = (x - \bar{x}) / \sigma$; valores com $|z| > 3$ (a mais de três desvios-padrão da média) são considerados *outliers*.

In [ ]:
dados_exemplo = np.array([0, 10, 12, 13, 15, 16, 18, 20, 22, 25, 30, 35, 40, 50, 350])

z_scores = (dados_exemplo - np.mean(dados_exemplo)) / np.std(dados_exemplo)
limite = 3
print('Outliers encontrados:', dados_exemplo[np.abs(z_scores) > limite])

## 9. Variáveis categóricas
Para modelos de *machine learning*, as variáveis binárias (sim/não, masculino/feminino) são mapeadas para 0/1 e as demais categóricas viram variáveis *dummy* com `pd.get_dummies`. O identificador do cliente não é uma variável explicativa e é removido.

In [ ]:
df.drop('id_cliente', axis=1, inplace=True)
mostrar_valores_unicos(df)

In [ ]:
mapeamento = {
    'nao': 0,
    'sim': 1,
    'masculino': 0,
    'feminino': 1
}
colunas = ['cliente.parceiro', 'Churn', 'cliente.dependentes', 'telefone.servico_telefone',
           'conta.faturamente_eletronico', 'cliente.genero']
df[colunas] = df[colunas].replace(mapeamento)

df = pd.get_dummies(df, dtype=int)
df.columns